In [0]:
import time

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DateType, IntegerType


def _valor(c):
    """trim + string vazia vira NULL (ausencia uniforme na silver)"""
    return F.nullif(F.trim(c), F.lit(""))


# Início da contagem do tempo total de execução da célula
_t0 = time.time()

# ----------------------------------------------------------------
# 1. LEITURA UNICA da bronze
# ----------------------------------------------------------------
bronze = spark.read.table("vidaplus.bronze.fact_sinan_sivep_notificacoes")

# ----------------------------------------------------------------
# 2. CHECK DE CONTRATO: PK no padrao NOT + 9 digitos (dicionario de dados)
#    Linha com PK invalida nao entra no ranking (a chave nao e confiavel
#    para agrupar) e vai direto para a quarentena com 'pk_fora_do_padrao'.
# ----------------------------------------------------------------
pk_valida = F.coalesce(F.trim(F.col("id_notificacao")).rlike(r"^NOT\d{9}$"), F.lit(False))

validas = bronze.filter(pk_valida)
invalidas_pk = bronze.filter(~pk_valida)  # PK NULL/nao-null-mas-fora-do-padrao cai aqui (coalesce -> False)

# ----------------------------------------------------------------
# 3. RANKING por PK + CLASSIFICACAO: revisao vs colisao
#    _rn = 1 e a versao mais recente por id_notificacao (desempate: arquivo)
# ----------------------------------------------------------------
w_pk = Window.partitionBy(F.trim(F.col("id_notificacao")))
w_order = w_pk.orderBy(F.col("_ingerido_em").desc(), F.col("_arquivo_origem").desc())

# Campos de IDENTIDADE (quem, quando, onde, qual agravo).
# Mudou entre versoes -> COLISAO: dois eventos diferentes com a mesma PK.
# Campos de STATUS (classificacao_final, criterio_confirmacao, evolucao,
# hospitalizacao, data_encerramento) podem mudar numa REVISAO legitima do SINAN.
CAMPOS_IDENTIDADE = [
    "data_inicio_sintomas",   # quando
    "data_notificacao",       # quando
    "sexo",                   # quem
    "idade",                  # quem
    "faixa_etaria",           # quem
    "id_bairro_residencia",   # onde
    "cod_municipio_ibge",     # onde
    "agravo",                 # qual agravo
    "cid10",                  # qual agravo
]

ranked = (
    validas
    .withColumn("_rn", F.row_number().over(w_order))
    .withColumn("_cnt_dup", F.count(F.lit(1)).over(w_pk))
)

# Traz o valor de identidade da VENCEDORA (_rn = 1) para todas as linhas da PK
for campo in CAMPOS_IDENTIDADE:
    ranked = ranked.withColumn(
        f"_win_{campo}",
        F.max(F.when(F.col("_rn") == 1, F.trim(F.col(campo)))).over(w_pk),
    )

# Compara cada linha com a vencedora (null-safe: NULL <=> NULL = igual).
# Nota: cada perdedora e comparada contra a VENCEDORA, nao contra a versao
# anterior da cadeia de revisoes (simplificacao aceitavel p/ cadeias curtas).
diff_cols = [
    F.when(~F.trim(campo).eqNullSafe(F.col(f"_win_{campo}")), F.lit(1)).otherwise(F.lit(0))
    for campo in CAMPOS_IDENTIDADE
]

ranked = (
    ranked
    .withColumn("_id_diferente", F.greatest(*diff_cols))  # 1 = identidade difere da vencedora
    .withColumn(
        "_pk_colisao",  # propaga p/ toda a particao: a vencedora tambem precisa saber
        F.max(
            F.when((F.col("_rn") > 1) & (F.col("_id_diferente") == 1), F.lit(1)).otherwise(F.lit(0))
        ).over(w_pk),
    )
    .withColumn(
        "_tipo_duplicata",
        F.when(F.col("_pk_colisao") == 1, F.lit("colisao"))
         .when(F.col("_rn") > 1, F.lit("revisao"))
         .otherwise(F.lit(None).cast("string")),
    )
    .cache()  # materializa uma vez: bronze e lida UMA vez nas duas escritas
)

# ----------------------------------------------------------------
# 4. SILVER: versao mais recente por PK, com transformacoes
#    PKs em COLISAO ficam de fora inteiras (todas as versoes vao p/ quarentena)
# ----------------------------------------------------------------
silver = (
    ranked
    .filter((F.col("_rn") == 1) & (F.col("_pk_colisao") == 0))
    .select(
        # IDs e codigos (STRING preserva zeros a esquerda)
        F.trim("id_notificacao").alias("id_notificacao"),  # nao-vazio garantido pelo check de PK
        _valor("cod_municipio_ibge").alias("cod_municipio_ibge"),
        _valor("id_bairro_residencia").alias("id_bairro_residencia"),
        _valor("id_unidade_notificante").alias("id_unidade_notificante"),

        # Localizacao
        _valor("municipio").alias("municipio"),
        _valor("uf").alias("uf"),
        # Datas (STRING YYYY-MM-DD -> DATE; NULL se ilegivel) — API nativa do Spark 4.0+
        F.col("data_inicio_sintomas").try_cast(DateType()).alias("data_inicio_sintomas"),
        F.col("data_notificacao").try_cast(DateType()).alias("data_notificacao"),
        F.col("data_encerramento").try_cast(DateType()).alias("data_encerramento"),

        # Numericos
        F.col("ano_epidemiologico").try_cast(IntegerType()).alias("ano_epidemiologico"),
        F.col("semana_epidemiologica").try_cast(IntegerType()).alias("semana_epidemiologica"),
        F.col("idade").try_cast(IntegerType()).alias("idade"),

        # Categoricos
        _valor("sistema_origem").alias("sistema_origem"),
        _valor("agravo").alias("agravo"),
        _valor("cid10").alias("cid10"),
        _valor("sexo").alias("sexo"),
        _valor("faixa_etaria").alias("faixa_etaria"),
        _valor("classificacao_final").alias("classificacao_final"),
        _valor("criterio_confirmacao").alias("criterio_confirmacao"),
        _valor("agente_etiologico_sorotipo").alias("agente_etiologico_sorotipo"),
        _valor("evolucao").alias("evolucao"),

        # Flag binaria ('0'/'1' -> INT)
        F.col("hospitalizacao").try_cast(IntegerType()).alias("hospitalizacao"),

        # Flag de qualidade: a PK tinha duplicata na bronze?
        F.when(F.col("_cnt_dup") > 1, F.lit(1)).otherwise(F.lit(0)).alias("flag_duplicata_pk"),

        # Metadados de ingestao
        F.col("_arquivo_origem"),
        F.col("_ingerido_em"),
    )
)
silver.explain(mode="formatted")  # debug: descomente p/ ver o plano de execucao
silver.write.mode("overwrite").saveAsTable("vidaplus.silver.fact_sinan_sivep_notificacoes")

# ----------------------------------------------------------------
# 5. QUARENTENA: perdedoras de revisao + PKs em colisao + PKs invalidas
#    (dado BRUTO, sem transformacao)
# ----------------------------------------------------------------
quarentena_duplicadas = (
    ranked
    .filter((F.col("_rn") > 1) | (F.col("_pk_colisao") == 1))
    .select(
        F.col("_rn"),                                     # qual versao era (2a, 3a... ou vencedora em colisao)
        F.col("_cnt_dup").alias("_total_duplicatas_pk"),   # quantas linhas tinham essa PK
        F.col("_tipo_duplicata"),                          # 'colisao' | 'revisao'
        *bronze.columns,                                   # colunas originais da bronze, sem cast
        F.when(F.col("_pk_colisao") == 1, F.lit("pk_colisao"))
         .otherwise(F.lit("pk_duplicada"))
         .alias("_regra_violada"),
        F.current_timestamp().alias("_data_quarentena"),
    )
)

# PKs fora do padrao: nunca foram ranqueadas (colunas de ranking ficam NULL)
quarentena_invalidas_pk = invalidas_pk.select(
    F.lit(None).cast("int").alias("_rn"),
    F.lit(None).cast("long").alias("_total_duplicatas_pk"),
    F.lit(None).cast("string").alias("_tipo_duplicata"),
    *bronze.columns,
    F.lit("pk_fora_do_padrao").alias("_regra_violada"),
    F.current_timestamp().alias("_data_quarentena"),
)

quarentena = quarentena_duplicadas.unionByName(quarentena_invalidas_pk)

quarentena.write.mode("overwrite").saveAsTable("vidaplus.silver.quarentena_notificacoes_pk_duplicada")

# ----------------------------------------------------------------
# 6. LOG da carga
# ----------------------------------------------------------------
n_bronze = bronze.count()
n_silver = silver.count()
n_quarentena = quarentena.count()
n_colisao = quarentena.filter(F.col("_tipo_duplicata") == "colisao").count()
n_revisao = quarentena.filter(F.col("_tipo_duplicata") == "revisao").count()
n_pk_invalida = quarentena.filter(F.col("_regra_violada") == "pk_fora_do_padrao").count()

print(f"Bronze:      {n_bronze:>8,} linhas ({n_pk_invalida:,} com PK fora do padrao)")
print(f"Silver:      {n_silver:>8,} linhas (mais recente por PK, sem PKs em colisao)")
print(f"Quarentena:  {n_quarentena:>8,} linhas | revisao: {n_revisao:,} | colisao: {n_colisao:,} | pk_invalida: {n_pk_invalida:,}")
print(f"Taxa de rejeicao: {n_quarentena / n_bronze:.2%}")
# Invariante de nao-perda: toda linha da bronze esta na silver OU na quarentena
assert n_silver + n_quarentena == n_bronze, (
    f"PERDA DE LINHAS: bronze={n_bronze}, silver+quarentena={n_silver + n_quarentena}"
)
tempo = time.time() - _t0  # tempo total de execução da célula
print(f"Tempo de Execução: {tempo:.2f} segundos")
ranked.unpersist()  # libera o cache